# PPE 탐지 4차 학습 — 라벨 의미 정합 + 홀드아웃 예약 (yolo26)

3차에서 **안전모 검출은 목표를 달성**했으나(helmet R 0.633 → 0.821), 두 가지 문제가 남았다.

## 4차가 바꾸는 것

| | 3차 | 4차 | 이유 |
|---|---|---|---|
| SH17 `head`·`helmet` | 사용 | **제거** | 다른 출처와 의미가 충돌한다(겹침 78.2%) |
| `SH17_FRACTION` | 0.3 | **0.5** | `foot` 499개로는 학습 불가였다. head·helmet 을 빼서 올려도 안전하다 |
| `HHW_FRACTION` | 0.5 | **0.15** | helmet 편중(1:0.33) · person 라벨 없어 평가 오염 |
| `E63_FRACTION` | 0.2 | **0.4** | 배포 도메인이고 라벨이 배타적이다 |
| `EPOCHS` | 20 | **30** | 20에포크에서 손실이 끝까지 하락 중이었다 |
| 홀드아웃 | 없음 | **시퀀스 12개 예약** | 3차는 채점이 불가능했다 |

## ★ SH17 라벨 의미 충돌 (4차의 핵심)

| 출처 | `head` | `helmet` | helmet 중 head 와 겹침 |
|---|---|---|---|
| E63 | 안전모 **미착용** 머리 | 안전모 **착용** 머리 | 0% (공존 없음) |
| CSS | NO-Hardhat | Hardhat | 0.0% |
| HHW | 헬멧 없는 머리 | 헬멧 쓴 머리 | 0.7% |
| **SH17** | **머리 (착용 여부 무관)** | **안전모 물체 자체** | **78.2%** |

SH17만 같은 머리에 둘을 동시에 붙인다. 박스 정의도 달라 서로를 오탐·미탐으로 만든다.
**1차 실패(교차 홀드아웃 `head` P 0.181)의 원인이 개수 불균형이 아니라 이것일 수 있다.**

## ★ 홀드아웃을 학습 전에 예약한다

3차는 프레임 단위로 20%를 뽑아 **43개 시퀀스 전부가 학습에 노출**됐다.
1차 홀드아웃이던 HHW도 학습에 편입해 채점할 시험지가 사라졌다.

4차는 **시퀀스를 통째로 떼어** 학습에 단 한 프레임도 넣지 않는다.

## 통합 8클래스

`person head helmet safety_vest foot shoes hands gloves`

미착용은 클래스로 두지 않는다. `head`(맨머리) 검출로 코드에서 판정한다.

⚠️ SH17(CC BY-NC-SA) · AI Hub(학습용 한정) 포함 — 산출 가중치에 `_nc` 가 붙는다. 상용 빌드 반입 금지.


## 1. 런타임 확인

`런타임 → 런타임 유형 변경 → T4 GPU`. **맨 위부터 순서대로 실행할 것.**

In [ ]:
!nvidia-smi
!free -g | head -2
import sys; print('python', sys.version.split()[0])

In [ ]:
%pip -q install "ultralytics>=8.4.0" kaggle roboflow
import ultralytics; ultralytics.checks()

## 2. 설정

### 예상 소요 (A100 · 자동 튜닝 적용 시)

3차 실측(13,719장 · batch 16 · workers 2 에서 5.7분/에포크) 기준 환산이다.
4차는 GPU 자동 튜닝이 적용돼 batch 64 · workers 8 로 돌므로 처리량이 약 2배다.

| 구성 | SH17/CSS/HHW/E63 | train | 에포크당 | 학습 | 총계 | helmet:head |
|---|---|---|---|---|---|---|
| **기본** | **0.5 / 1.0 / 0.15 / 0.4** | 약 17,200 | 약 3.6분 | 1.8h | **약 2.5h** | 1 : 0.69 |
| 가벼움 | 0.4 / 1.0 / 0.15 / 0.3 | 약 13,700 | 약 2.9분 | 1.5h | 약 2.1h | 1 : 0.68 |
| 넉넉 | 0.6 / 1.0 / 0.2 / 0.6 | 약 23,000 | 약 4.8분 | 2.4h | 약 3.2h | 1 : 0.70 |

T4 라면 5~6배 걸린다. 컴퓨팅 단위를 먼저 확인할 것.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ---- 설정 -------------------------------------------------------------
MODEL   = 'yolo26s.pt'
IMGSZ   = 640          # 1·3차 검증됨. 960 은 에포크당 31분이라 완주 불가였다
EPOCHS  = 30           # 3차 20에포크는 손실이 끝까지 하락 중이라 부족했다
BATCH   = None         # None = GPU 자동 감지(아래). 고정하려면 숫자 지정
WORKERS = None         # None = GPU 자동 감지. 데이터 공급 병목을 좌우한다
RESUME  = False        # 세션 끊긴 뒤 이어서 학습할 때만 True (다른 값은 건드리지 말 것)

# 출처별 비율 (위 표 참조)
SH17_FRACTION = 0.5    # 3차 0.3 에서 foot 499개로 학습 불가였다.
                       # head·helmet 매핑을 뺐으므로 올려도 균형을 해치지 않는다
CSS_FRACTION  = 1.0    # safety_vest 유일 출처. 규모가 작아 전량
HHW_FRACTION  = 0.15   # 3차 0.5 는 helmet 편중(1:0.33)이 심했다. person 라벨도 없어
                       # 평가를 오염시키므로 helmet 물량만 얻을 만큼만 쓴다
E63_FRACTION  = 0.4    # ★ 배포 도메인. 라벨이 배타적이라 비중을 올린다

# ★ 홀드아웃 예약 — 학습 전에 시퀀스를 통째로 뗀다.
#   3차는 프레임 단위로 나눠 43개 시퀀스 전부가 학습에 노출돼 채점이 불가능했다.
E63_HOLDOUT_SEQ_RATIO = 0.30   # 시퀀스 43개 중 12개. 0.18 이면 홀드아웃이 420장뿐이라 부족했다

USE_SH17 = True        # ⚠️ CC BY-NC-SA(비상업)
USE_CSS  = True        # CC BY 4.0
USE_HHW  = True
USE_E63  = True        # ⚠️ AI Hub — 학습용 한정(상업 이용은 수행기관 별도 협의)
# -----------------------------------------------------------------------

NC = USE_SH17 or USE_E63        # 비상업 데이터 포함 여부
RUN_NAME  = f"{Path(MODEL).stem}_{IMGSZ}_v4" + ("_nc" if NC else "")
DRIVE     = Path('/content/drive/MyDrive/vitalguard-ppe')
RUNS      = DRIVE / 'runs'
E63_TAR   = DRIVE / 'datasets' / 'e63_yolo.tar'   # 맥에서 변환해 올린 단일 아카이브
E63_SRC   = Path('/content/e63_yolo')             # 여기로 풀어서 쓴다(로컬 디스크)
RAW       = Path('/content/raw')
DATA      = Path('/content/ppe3')
DATA_YAML = DATA / 'data.yaml'
for p in (DRIVE, RUNS, RAW, DATA):
    p.mkdir(parents=True, exist_ok=True)

# ---- GPU 자동 튜닝 -----------------------------------------------------
# 1차(T4)에서 GPU 가 71% 놀았다. 원인은 T4 인스턴스의 CPU 2코어가 이미지 공급을
# 못 따라간 것. A100/L4 인스턴스는 CPU 코어가 많아 workers 를 올려야 성능이 난다.
import os, torch
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
_cpu = os.cpu_count() or 2
if BATCH is None or WORKERS is None:
    if 'A100' in _gpu:      _b, _w = 64, min(8, _cpu)
    elif 'H100' in _gpu:    _b, _w = 96, min(8, _cpu)
    elif 'L4' in _gpu:      _b, _w = 32, min(6, _cpu)
    elif 'V100' in _gpu:    _b, _w = 32, min(4, _cpu)
    else:                   _b, _w = 16, min(2, _cpu)      # T4 등
    BATCH   = _b if BATCH   is None else BATCH
    WORKERS = _w if WORKERS is None else WORKERS
print(f'GPU  : {_gpu} · CPU {_cpu}코어 → batch {BATCH} · workers {WORKERS}')
if 'A100' in _gpu or 'H100' in _gpu:
    print('       ★ 컴퓨팅 단위 소모가 T4 의 약 6배다. 남은 단위를 확인할 것')
# -----------------------------------------------------------------------

print('run  :', RUN_NAME)
print('E63  :', E63_TAR, '→',
      f'{E63_TAR.stat().st_size/1e9:.2f} GB' if E63_TAR.exists() else '⚠️ 없음(Drive 동기화 확인)')
if NC:
    print('\n⚠️  비상업 데이터 포함 — 산출 가중치는 상용 빌드에 반입하지 말 것(-nc 접미사 유지).')

## 3. 데이터셋 다운로드

- **SH17** — Kaggle. **API 토큰 문자열** 필요(`kaggle.com/settings` → API → Create New Token, **한 번만** 표시)
- **CSS · HHW** — Roboflow. API 키 필요
- **E63** — 다운로드 없음. Drive 에 이미 YOLO 형식으로 있다

In [ ]:
def _ensure(pkg, mod=None):
    """패키지가 없으면 설치한다 — 설치 셀을 건너뛰어도 동작하도록.

    ★ import 하지 않고 존재만 확인한다. kaggle 은 import 시점에 인증을 시도해
      자격증명이 없으면 그 자리에서 죽는다(exit(1) → NameError).
    """
    import importlib, importlib.util, subprocess, sys
    importlib.invalidate_caches()
    if importlib.util.find_spec(mod or pkg) is None:
        print(f'{pkg} 설치 중...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
        importlib.invalidate_caches()


if USE_SH17:
    import os
    from getpass import getpass

    # 자격증명을 먼저 만든 뒤 kaggle 을 부른다 — 순서가 뒤바뀌면 인증 오류로 죽는다
    tok = getpass('Kaggle API token: ').strip()
    os.environ['KAGGLE_API_TOKEN'] = tok
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'access_token').write_text(tok)
    (kdir / 'access_token').chmod(0o600)
    del tok

    _ensure('kaggle')
    !kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection -p /content/raw --unzip -q
    print('SH17 다운로드 완료')

# 403 이면 브라우저에서 데이터셋 페이지를 열고 Download 를 눌러 규정에 동의할 것

In [ ]:
_ensure('roboflow')
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))

if USE_CSS:
    css = (rf.workspace('roboflow-universe-projects')
             .project('construction-site-safety')
             .version(27).download('yolov8', location='/content/raw/css'))
    print('CSS →', css.location)

if USE_HHW:
    # v13 이 helmet/head 분리 버전(v14 는 한 클래스로 병합돼 있어 못 씀)
    hhw = (rf.workspace('joseph-nelson')
             .project('hard-hat-workers')
             .version(13).download('yolov8', location='/content/raw/hhw'))
    print('HHW →', hhw.location)

In [ ]:
# ★ 실제 구조와 클래스 순서를 눈으로 확인한다. 배포본마다 다르므로 반드시 대조할 것.
!find /content/raw -maxdepth 3 -type d | head -20
print()
for y in sorted(Path('/content/raw').rglob('*.yaml')):
    print('---', y)
    print(y.read_text()[:260])
print('--- E63 (Drive, 변환 완료본) ---')
print((E63_SRC / 'data.yaml').read_text() if (E63_SRC / 'data.yaml').exists() else '없음')

## 4. 통합 8클래스로 리매핑

```
0 person   1 head   2 helmet   3 safety_vest
4 foot     5 shoes  6 hands    7 gloves
```

미착용은 클래스로 두지 않고 `head`(맨머리) + person 포함관계로 코드에서 판정한다.

**처리 세 가지**

- **증강 사본 제거** — Roboflow 는 `<원본>_jpg.rf.<해시>` 로 증강본을 만든다. 원본 1장만 남긴다
- **출처별 서브샘플** — 파일명 해시로 결정적으로 고른다(재실행·재개해도 동일)
- **E63 는 이미 YOLO** — 변환 없이 비율만 적용해 연결한다

In [ ]:
import hashlib, re, shutil, yaml
from collections import Counter
from pathlib import Path

UNIFIED = ['person', 'head', 'helmet', 'safety_vest', 'foot', 'shoes', 'hands', 'gloves']
UNIFIED_IDX = {n: i for i, n in enumerate(UNIFIED)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def _norm(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


# ★ SH17 의 head·helmet 은 뺀다 — 다른 출처와 의미가 다르다.
#   SH17 은 머리에 head 를, 안전모에 helmet 을 따로 붙여 둘이 78.2% 겹친다.
#   E63·CSS·HHW 는 '미착용 머리 = head / 착용 머리 = helmet' 로 배타적이다.
SH17_MAP = {'person': 'person', 'safetyvest': 'safety_vest', 'foot': 'foot',
            'shoes': 'shoes', 'hands': 'hands', 'gloves': 'gloves'}

# 출처마다 라벨링한 클래스가 다르다. 라벨이 없는 출처의 이미지에서는 정탐도 오탐으로
# 계산되므로(3차에서 person P 0.493 로 찍혔으나 실제는 0.891) 평가 때 대상을 제한한다.
SOURCE_CLASSES = {
    'sh17': {'person', 'foot', 'shoes', 'hands', 'gloves', 'safety_vest'},
    'css':  {'person', 'head', 'helmet', 'safety_vest'},
    'hhw':  {'head', 'helmet'},
    'e63':  {'head', 'helmet'},
}
CSS_MAP = {'person': 'person', 'hardhat': 'helmet', 'nohardhat': 'head',
           'safetyvest': 'safety_vest'}
HHW_MAP = {'helmet': 'helmet', 'head': 'head', 'person': 'person'}

# ★ SH17 공식 클래스 순서 — sh17.yaml 원문. 논문/README 나열 순서와 다르다.
SH17_NAMES = ['person', 'ear', 'ear-mufs', 'face', 'face-guard', 'face-mask', 'foot',
              'tool', 'glasses', 'gloves', 'helmet', 'hands', 'head', 'medical-suit',
              'shoes', 'safety-suit', 'safety-vest']

RF_AUG = re.compile(r'^(.*?)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+$', re.I)


def rf_base(stem):
    """증강 사본들의 공통 원본 이름. 증강본이 아니면 그대로 반환."""
    m = RF_AUG.match(stem)
    return m.group(1) if m else stem


def load_names(data_yaml):
    d = yaml.safe_load(Path(data_yaml).read_text())['names']
    return [d[k] for k in sorted(d)] if isinstance(d, dict) else list(d)


def find_yaml(root, exclude=(), prefer=('sh17.yaml', 'data.yaml')):
    cands = [p for p in Path(root).rglob('*.yaml') if not any(e in p.parts for e in exclude)]
    for name in prefer:
        for p in cands:
            if p.name == name:
                return p
    return cands[0] if cands else None


def build_index_map(src_names, name_map):
    out = {}
    for i, n in enumerate(src_names):
        tgt = name_map.get(_norm(n))
        out[i] = UNIFIED_IDX[tgt] if tgt else None
    return out


def _hash1000(s):
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 1000


def deterministic_split(stem, val_ratio=0.1):
    return 'val' if _hash1000(stem) < val_ratio * 1000 else 'train'


def _place(img, lines, dst_root, sp, stem, link=True):
    """이미지 링크 + 라벨 파일 기록."""
    io_ = Path(dst_root) / 'images' / sp
    lo_ = Path(dst_root) / 'labels' / sp
    io_.mkdir(parents=True, exist_ok=True)
    lo_.mkdir(parents=True, exist_ok=True)
    di = io_ / (stem + img.suffix)
    if not di.exists():
        if link:
            try:
                di.symlink_to(img.resolve())
            except OSError:
                shutil.copy2(img, di)
        else:
            shutil.copy2(img, di)
    (lo_ / (stem + '.txt')).write_text('\n'.join(lines) + '\n')


def collect_pairs(root, val_ratio=0.1, dedupe_aug=True, fraction=1.0):
    """(이미지, 라벨, split) 목록. 증강 사본 제거·서브샘플 적용."""
    root = Path(root)
    seen, pairs, n_aug, n_frac = set(), [], 0, 0
    for img in sorted(root.rglob('*')):
        if img.suffix.lower() not in IMG_EXT or not img.is_file() or img.name.startswith('.'):
            continue   # 숨김·AppleDouble(._*) 제외
        pl = [p.lower() for p in img.parts]
        if 'images' not in pl:
            continue
        base = rf_base(img.stem)
        if dedupe_aug:
            if base in seen:
                n_aug += 1
                continue
            seen.add(base)
        if fraction < 1.0 and _hash1000('frac' + base) >= fraction * 1000:
            n_frac += 1
            continue
        i = len(pl) - 1 - pl[::-1].index('images')
        lbl = Path(*img.parts[:i], 'labels', *img.parts[i + 1:]).with_suffix('.txt')
        s = set(pl)
        sp = ('train' if 'train' in s else
              'val' if s & {'val', 'valid'} else
              'test' if 'test' in s else
              deterministic_split(base, val_ratio))
        pairs.append((img, lbl, sp))
    return pairs, n_aug, n_frac


def remap(pairs, dst_root, idx_map, prefix, link=True):
    """원본 라벨을 통합 인덱스로 다시 쓰고 이미지를 모은다."""
    kept = dropped = 0
    counts = Counter()
    for img, lbl, sp in pairs:
        lines = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                try:
                    new = idx_map.get(int(float(parts[0])))
                except ValueError:
                    continue
                if new is None:
                    continue
                lines.append(' '.join([str(new)] + parts[1:5]))
                counts[UNIFIED[new]] += 1
        if not lines:
            dropped += 1
            continue
        _place(img, lines, dst_root, sp, f'{prefix}_{img.stem}', link)
        kept += 1
    return kept, dropped, counts


def seq_of(stem):
    """E63 프레임 이름에서 촬영 시퀀스 키를 뽑는다.

    e63_E63_DATA3_H1_L5_D2023-10-26-11-55_082_001290 → ..._082
    같은 시퀀스의 인접 프레임은 거의 동일하므로 분할은 반드시 시퀀스 단위로 한다.
    """
    return stem.rsplit('_', 1)[0]


def e63_reserve_seqs(src, ratio):
    """홀드아웃으로 예약할 시퀀스 집합을 결정적으로 고른다."""
    seqs = set()
    for sp in ('train', 'val'):
        d = Path(src) / 'images' / sp
        if not d.is_dir():
            continue
        for p in d.iterdir():
            if p.suffix.lower() in IMG_EXT and not p.name.startswith('.'):
                seqs.add(seq_of(p.stem))
    return seqs, {s for s in seqs if _hash1000('seq' + s) < ratio * 1000}


def link_yolo(src, dst_root, fraction=1.0, link=True, exclude_seqs=frozenset()):
    """이미 통합 8클래스 YOLO 형식인 데이터셋을 비율만 적용해 연결한다(E63 용).

    라벨 인덱스가 이미 통합 기준이므로 재매핑하지 않는다.
    exclude_seqs 에 든 시퀀스는 통째로 건너뛴다(홀드아웃 예약분).
    """
    src = Path(src)
    kept = skipped = held = 0
    counts = Counter()
    for sp in ('train', 'val', 'test'):
        idir = src / 'images' / sp
        if not idir.is_dir():
            continue
        for img in sorted(idir.iterdir()):
            if img.suffix.lower() not in IMG_EXT or img.name.startswith('.'):
                continue   # 숨김·AppleDouble(._*) 은 이미지가 아니다
            if seq_of(img.stem) in exclude_seqs:
                held += 1
                continue   # 홀드아웃 예약 시퀀스 — 학습에 절대 넣지 않는다
            if fraction < 1.0 and _hash1000('frac' + img.stem) >= fraction * 1000:
                skipped += 1
                continue
            lbl = src / 'labels' / sp / (img.stem + '.txt')
            if not lbl.exists():
                continue
            lines = [l for l in lbl.read_text().splitlines() if l.strip()]
            if not lines:
                continue
            for l in lines:
                counts[UNIFIED[int(l.split()[0])]] += 1
            _place(img, lines, dst_root, sp, img.stem, link)   # 이미 e63_ 접두사가 붙어 있음
            kept += 1
    return kept, skipped, held, counts


print('리매핑 유틸 로드 완료')

In [ ]:
SOURCES = []
if USE_SH17:
    y = find_yaml('/content/raw', exclude=('css', 'hhw'))
    print('SH17 클래스 정의:', y or '(동봉 yaml 없음 → 공식 하드코딩 사용)')
    SOURCES.append(dict(name='sh17', root='/content/raw', yaml=str(y) if y else None,
                        names=SH17_NAMES, mapping=SH17_MAP, fraction=SH17_FRACTION))
if USE_CSS:
    SOURCES.append(dict(name='css', root='/content/raw/css', yaml='/content/raw/css/data.yaml',
                        names=None, mapping=CSS_MAP, fraction=CSS_FRACTION))
if USE_HHW:
    SOURCES.append(dict(name='hhw', root='/content/raw/hhw', yaml='/content/raw/hhw/data.yaml',
                        names=None, mapping=HHW_MAP, fraction=HHW_FRACTION))

total = Counter()
for s in SOURCES:
    names = load_names(s['yaml']) if s['yaml'] else s['names']
    idx_map = build_index_map(names, s['mapping'])
    mapped = {names[i]: UNIFIED[v] for i, v in idx_map.items() if v is not None}
    print(f"\n[{s['name']}] 원본 {len(names)}클래스 → 유지 {len(mapped)}종  (비율 {s['fraction']})")
    print('   순서:', names)
    print('   매핑:', mapped)
    if len(mapped) < len(set(s['mapping'].values())):
        print(f"   ⚠️ 매핑 누락 {sorted(set(s['mapping'].values()) - set(mapped.values()))}")
    pairs, n_aug, n_frac = collect_pairs(s['root'], fraction=s['fraction'])
    print(f"   이미지 {len(pairs)}장  (증강사본 제외 {n_aug} · 서브샘플 제외 {n_frac})")
    kept, dropped, counts = remap(pairs, DATA, idx_map, s['name'])
    print(f'   유지 {kept} / 제외 {dropped}')
    print('  ', dict(counts))
    total.update(counts)

# ★ E63 — 이미 YOLO 형식이라 변환 없이 연결만
if USE_E63:
    # Drive 는 파일이 3만 개면 동기화가 매우 느리다 → 단일 tar 로 올려두고 여기서 푼다(약 1분)
    if not (E63_SRC / 'data.yaml').exists():
        assert E63_TAR.exists(), f'E63 아카이브 없음: {E63_TAR} — Drive 동기화 확인'
        E63_SRC.mkdir(parents=True, exist_ok=True)
        print(f'E63 아카이브 해제 중 ({E63_TAR.stat().st_size/1e9:.2f} GB)…')
        # macOS tar 가 넣은 애플 확장속성(com.apple.provenance)을 리눅스 tar 가 모른다.
        # 무해하지만 파일당 한 줄씩 3만 줄이 쏟아지므로 그 경고만 걸러낸다.
        !tar -xf "{E63_TAR}" -C "{E63_SRC}" 2>&1 | grep -v 'LIBARCHIVE.xattr' | head -20
        # 맥에서 만든 아카이브라 파일마다 AppleDouble(._*) 메타데이터가 딸려 있다.
        # 맥 tar 는 목록에서 감추지만 리눅스에서는 실제 파일로 풀리고, 확장자가
        # .jpg/.txt 라 라벨로 오인돼 UnicodeDecodeError 를 낸다.
        !find "{E63_SRC}" -name '._*' -delete
        !find "{E63_SRC}" -name '.DS_Store' -delete
        print('  해제 완료')
    assert (E63_SRC / 'data.yaml').exists(), f'E63 해제 실패: {E63_SRC}'
    # ★ 홀드아웃 시퀀스를 먼저 예약한다 — 학습에 한 프레임도 넣지 않는다
    ALL_SEQ, HOLD_SEQ = e63_reserve_seqs(E63_SRC, E63_HOLDOUT_SEQ_RATIO)
    print(f'\n시퀀스 {len(ALL_SEQ)}개 중 {len(HOLD_SEQ)}개를 홀드아웃으로 예약')
    assert 3 <= len(HOLD_SEQ) < len(ALL_SEQ), 'E63_HOLDOUT_SEQ_RATIO 를 조정할 것'
    # tar 를 /content 에 풀었으므로 이미 로컬 디스크다 → 링크로 충분(복사 불필요)
    kept, skipped, held, counts = link_yolo(E63_SRC, DATA, fraction=E63_FRACTION,
                                            link=True, exclude_seqs=HOLD_SEQ)
    print(f"[e63] 국내 AI Hub (비율 {E63_FRACTION})")
    print(f'   유지 {kept} / 서브샘플 제외 {skipped} / 홀드아웃 예약 {held}')
    print('  ', dict(counts))
    total.update(counts)

print('\n=== 통합 데이터셋 클래스별 인스턴스 ===')
for n in UNIFIED:
    c = total.get(n, 0)
    flag = '  ⚠️ 부족' if 0 < c < 500 else ('  ⚠️ 없음' if c == 0 else '')
    print(f'  {n:<12} {c:>7,}{flag}')

if total.get('head') and total.get('helmet'):
    print(f"\n★ helmet : head = 1 : {total['head']/total['helmet']:.2f}")
    print('   1차 1:3.2 · 3차 1:0.75 → 4차 목표 1:0.7~0.8')
    print('   ⚠️ SH17 은 head·helmet 을 기여하지 않는다(라벨 의미 충돌로 제외)')

In [ ]:
splits = {p.name: len(list(p.glob('*'))) for p in (DATA / 'images').iterdir() if p.is_dir()}
print('split 별 이미지 수:', splits)

cfg = {'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
       'names': {i: n for i, n in enumerate(UNIFIED)}}
if 'test' in splits:
    cfg['test'] = 'images/test'
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print()
print(DATA_YAML.read_text())

est = splits.get('train', 0) / 11521 * 29.6      # 1차 실측 환산
print(f'\n예상 소요 — 에포크당 약 {est:.0f}분 · {EPOCHS} 에포크 총 {est*EPOCHS/60:.1f}시간')
if est * EPOCHS / 60 > 15:
    print('⚠️ 15시간 초과 — 출처별 FRACTION 을 낮추거나 EPOCHS 를 줄일 것')

In [ ]:
# ★ 예약된 시퀀스로 홀드아웃을 만든다 — 학습에 단 한 프레임도 쓰이지 않았다
from collections import defaultdict

E63_HOLD = Path('/content/holdout_e63_seq')
HOLD_PER_SEQ = 200         # 시퀀스당 최대 장수. 인접 프레임은 거의 같지만
                           # 인스턴스 수가 적으면 신뢰구간이 넓어져 판정을 못 한다

if USE_E63:
    if E63_HOLD.exists():
        shutil.rmtree(E63_HOLD)
    (E63_HOLD / 'images' / 'val').mkdir(parents=True)
    (E63_HOLD / 'labels' / 'val').mkdir(parents=True)

    by_seq = defaultdict(list)
    for sp in ('train', 'val'):
        d = E63_SRC / 'images' / sp
        if not d.is_dir():
            continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in IMG_EXT and not p.name.startswith('.') \
                    and seq_of(p.stem) in HOLD_SEQ:
                by_seq[seq_of(p.stem)].append(p)

    n = 0
    for s_, ps in sorted(by_seq.items()):
        ps = sorted(ps)
        for p in ps[::max(1, len(ps) // HOLD_PER_SEQ)][:HOLD_PER_SEQ]:
            lbl = p.parents[2] / 'labels' / p.parts[-2] / (p.stem + '.txt')
            if not lbl.exists():
                continue
            (E63_HOLD / 'images' / 'val' / p.name).symlink_to(p.resolve())
            shutil.copy2(lbl, E63_HOLD / 'labels' / 'val' / lbl.name)
            n += 1

    (E63_HOLD / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(E63_HOLD), 'train': 'images/val', 'val': 'images/val',
         'names': {i: x for i, x in enumerate(UNIFIED)}},
        allow_unicode=True, sort_keys=False))

    # ★ 누수 검사 — 홀드아웃이 학습셋과 한 장도 겹치면 안 된다
    tr = {p.stem for sp in ('train', 'val')
          for p in (DATA / 'images' / sp).glob('e63_*')}
    ho = {p.stem for p in (E63_HOLD / 'images' / 'val').iterdir()}
    assert not (tr & ho), f'홀드아웃 누수 {len(tr & ho)}장 — 예약 로직 확인'

    cnt = Counter()
    for f in (E63_HOLD / 'labels' / 'val').glob('*.txt'):
        for l in f.read_text().splitlines():
            if l.strip():
                cnt[UNIFIED[int(l.split()[0])]] += 1
    print(f'홀드아웃 시퀀스 {len(by_seq)}개 → {n:,}장 (학습 미노출)')
    print('  클래스:', dict(cnt))
    print('  누수 검사: 통과 (학습셋과 교집합 0)')


## 5. 학습

체크포인트는 Drive 에 저장된다. 세션이 끊기면 설정 셀에서 **`RESUME = True` 로만** 바꾸고
**설치 → 설정 → 데이터 → 리매핑 → 학습** 순서로 다시 실행한다(다른 값은 건드리지 말 것).

**학습 중 다른 셀을 실행하지 말 것** — 데이터 폴더를 건드리면 학습이 깨진다.

**화면의 ETA(`n:nn<n:nn`)는 무시할 것.** 순간 속도라 정체를 반영하지 못해 실제보다 빠르게 나온다.
완료된 에포크의 실제 소요로만 판단한다.

In [ ]:
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'
if RESUME and last.exists():
    print('재개:', last)
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=str(RUNS), name=RUN_NAME, exist_ok=True,
        patience=30, workers=WORKERS, seed=0, plots=True,
    )

best = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('\nbest:', best)

## 6. 평가 — 1층 검출 지표

1차 수치를 표에 병기한다. **`helmet` recall 이 0.633 보다 올랐는지**가 1차 목표다.

In [ ]:
import pandas as pd
from ultralytics import YOLO


def per_class(weights, data_yaml=None, imgsz=None, split='val'):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml or DATA_YAML), imgsz=imgsz or IMGSZ, split=split, verbose=False)
    rows = [{'class': r.names[int(c)],
             'P': round(float(r.box.p[i]), 3), 'R': round(float(r.box.r[i]), 3),
             'mAP50': round(float(r.box.ap50[i]), 3), 'mAP50-95': round(float(r.box.ap[i]), 3)}
            for i, c in enumerate(r.box.ap_class_index)]
    return pd.DataFrame(rows).sort_values('R'), r


def _val_subset(prefixes, tag):
    """val 을 출처별로 잘라 임시 데이터셋을 만든다."""
    sub = Path(f'/content/val_{tag}')
    if sub.exists():
        shutil.rmtree(sub)
    (sub / 'images' / 'val').mkdir(parents=True)
    (sub / 'labels' / 'val').mkdir(parents=True)
    n = 0
    for p in sorted((DATA / 'images' / 'val').iterdir()):
        if p.name.split('_')[0] not in prefixes:
            continue
        lbl = DATA / 'labels' / 'val' / (p.stem + '.txt')
        if not lbl.exists():
            continue
        (sub / 'images' / 'val' / p.name).symlink_to(p.resolve())
        shutil.copy2(lbl, sub / 'labels' / 'val' / lbl.name)
        n += 1
    (sub / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(sub), 'train': 'images/val', 'val': 'images/val',
         'names': {i: x for i, x in enumerate(UNIFIED)}},
        allow_unicode=True, sort_keys=False))
    return sub, n


def fair_eval(weights):
    """클래스마다 그 클래스를 라벨링한 출처의 이미지에서만 평가한다.

    라벨이 없는 출처의 이미지에서는 올바른 검출도 오탐으로 계산된다.
    3차에서 person P 가 0.493 으로 찍혔으나 이 방식으로는 0.891 이었다.
    """
    groups = {}
    for cls in UNIFIED:
        pref = frozenset(s for s, cs in SOURCE_CLASSES.items() if cls in cs)
        if pref:
            groups.setdefault(pref, []).append(cls)
    out = []
    for pref, classes in groups.items():
        sub, n = _val_subset(pref, '_'.join(sorted(pref)))
        print(f'  {sorted(classes)}  ←  {sorted(pref)} val {n}장')
        df, _ = per_class(weights, data_yaml=sub / 'data.yaml')
        out.append(df[df['class'].isin(classes)])
    return pd.concat(out).sort_values('class').reset_index(drop=True)


print('=== 공정 평가 (클래스별로 라벨이 있는 출처에서만) ===')
df = fair_eval(best)

# 3차 참고값 (P, R). head·helmet 은 전체 val, 나머지는 sh17+css 기준이라 조건이 완전히 같지는 않다.
V3 = {'person': (0.891, 0.746), 'head': (0.673, 0.835), 'helmet': (0.813, 0.821),
      'safety_vest': (0.776, 0.302), 'foot': (0.000, 0.000), 'shoes': (0.535, 0.408),
      'hands': (0.764, 0.734), 'gloves': (0.484, 0.459)}
df['P_3차'] = df['class'].map(lambda c: V3.get(c, (None, None))[0])
df['R_3차'] = df['class'].map(lambda c: V3.get(c, (None, None))[1])
df['R_변화'] = (df['R'] - df['R_3차']).round(3)
display(df[['class', 'P', 'P_3차', 'R', 'R_3차', 'R_변화', 'mAP50', 'mAP50-95']])
print('\n★ head P 상승 + helmet 유지 가 4차의 성공 조건.')
print('   foot 은 3차에서 P 0 · R 0 이었다. SH17 비중을 올렸으니 0 을 벗어나는지 본다.')


## 7. 평가 — 2층 착용 판정

`judge_ppe()` 는 person 박스를 요구하는데 **E63·HHW 에는 person 라벨이 없다.**
그래서 대부분의 데이터에서 그 방식으로는 정확도를 잴 수 없다.

대신 **이미지 단위**로 잰다. E63·CSS·HHW 는 `head`(미착용)와 `helmet`(착용)이 배타적이라
정답이 명확하다.

```
정답   라벨에 head 박스가 있다   → 미착용자 존재
예측   모델이 head 를 검출했다   → 미착용 알림 발생
```

**판정 게이트와 직접 대응한다** — `안전모 미착용 판정 recall ≥ 0.90 @ precision ≥ 0.85`.


In [ ]:
HEAD_I = UNIFIED_IDX['head']


def bare_head_eval(weights, data_root, confs=(0.15, 0.25, 0.35, 0.45, 0.55)):
    """이미지 단위 '안전모 미착용' 알림의 정밀도·재현율을 임계별로 낸다."""
    m = YOLO(str(weights))
    root = Path(data_root)
    imgs = [p for p in sorted((root / 'images' / 'val').iterdir())
            if p.suffix.lower() in IMG_EXT]
    gt, pred = [], []
    for res in m.predict(imgs, imgsz=IMGSZ, conf=min(confs), stream=True, verbose=False):
        lbl = root / 'labels' / 'val' / (Path(res.path).stem + '.txt')
        gt.append(bool(lbl.exists()) and any(
            l.split() and int(l.split()[0]) == HEAD_I
            for l in lbl.read_text().splitlines() if l.strip()))
        pred.append([c for k, c in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist())
                     if int(k) == HEAD_I])
    rows = []
    for t in confs:
        hit = [any(c >= t for c in ps) for ps in pred]
        tp = sum(1 for g, h in zip(gt, hit) if g and h)
        fp = sum(1 for g, h in zip(gt, hit) if not g and h)
        fn = sum(1 for g, h in zip(gt, hit) if g and not h)
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        rows.append({'conf': t, 'P': round(P, 3), 'R': round(R, 3),
                     'F1': round(2 * P * R / (P + R), 3) if P + R else 0.0,
                     'TP': tp, 'FP': fp, 'FN': fn,
                     '게이트': '통과' if (R >= 0.90 and P >= 0.85) else ''})
    print(f'이미지 {len(gt):,}장 · 미착용자 있는 장면 {sum(gt):,}장')
    return pd.DataFrame(rows)


print('=== 인도메인 val 기준 ===')
display(bare_head_eval(best, DATA))
print('★ 게이트: recall ≥ 0.90 @ precision ≥ 0.85')
print('   여기는 학습 분포다. 진짜 판정은 다음 장의 홀드아웃에서 한다.')


## 8. 홀드아웃 — ★ 4차 성패 판정

학습 전에 예약해 둔 **E63 시퀀스**로 평가한다. 이 시퀀스의 프레임은 학습에 한 장도 쓰이지 않았다.

3차까지 쓰던 교차 데이터셋은 전부 소진됐다.

| 후보 | 상태 |
|---|---|
| HHW | 3차에서 학습에 편입 |
| CSS | 학습에 편입 |
| Roboflow `safety-helmet-and-reflection-vest` | `head` 클래스 없음 |
| Roboflow `helmet-detection-v3` | CSS 재업로드 |

**예약 홀드아웃이 유일하게 신뢰할 수 있는 채점 수단이다.**
같은 도메인이라 교차 데이터셋만큼 엄격하진 않지만, 배포 대상이 국내 현장이므로
오히려 목적에는 더 부합한다.


In [ ]:
assert E63_HOLD.exists(), '홀드아웃이 없다 — 4장의 홀드아웃 구성 셀을 먼저 실행할 것'

print('=== 1층 검출 (예약 시퀀스) ===')
df_ho, _ = per_class(best, data_yaml=E63_HOLD / 'data.yaml')
display(df_ho[df_ho['class'].isin(['head', 'helmet'])][['class', 'P', 'R', 'mAP50', 'mAP50-95']])

print('\n참고값')
print('  1차 교차 홀드아웃(HHW)   head P 0.181 · helmet P 0.524   ← 실패 판정의 근거')
print('  3차 누수 있는 E63 표본   head P 0.919 · helmet P 0.955   ← 상한선일 뿐 판정 불가')
print('  4차 (여기)               학습 미노출 시퀀스 — 신뢰할 수 있는 수치\n')

print('=== 2층 미착용 알림 판정 (예약 시퀀스) ===')
display(bare_head_eval(best, E63_HOLD))
print('★ 게이트: recall ≥ 0.90 @ precision ≥ 0.85')


In [ ]:
# 임계 스윕 — 운영에 쓸 conf 를 고른다
import numpy as np

sweep = bare_head_eval(best, E63_HOLD, confs=tuple(np.round(np.arange(0.10, 0.71, 0.05), 2)))
display(sweep)

ok = sweep[(sweep['R'] >= 0.90) & (sweep['P'] >= 0.85)]
if len(ok):
    pick = ok.sort_values('F1', ascending=False).iloc[0]
    print(f"\n★ 게이트 통과 — 권장 conf = {pick['conf']}  (P {pick['P']} · R {pick['R']})")
else:
    best_f1 = sweep.sort_values('F1', ascending=False).iloc[0]
    print(f"\n게이트 미달. 최고 F1 = {best_f1['F1']} @ conf {best_f1['conf']} "
          f"(P {best_f1['P']} · R {best_f1['R']})")
    print('   recall 이 부족하면 데이터·에포크, precision 이 부족하면 라벨 정합을 의심할 것')


## 9. 기준선 비교 — 현행 라이브 모델

판정 기준은 "얼마나 좋은가"가 아니라 **"현행보다 나은가"** 다.
현행 CAM-03 은 Hexmon `vyra_ppe.pt`(YOLOv8m 14클래스)를 쓴다.

**같은 홀드아웃에 현행 모델도 태워야** 공정한 비교가 된다.

In [ ]:
!curl -sL -o /content/vyra_ppe.pt https://huggingface.co/Hexmon/vyra-yolo-ppe-detection/resolve/main/best.pt
baseline = YOLO('/content/vyra_ppe.pt')
print('현행 모델 클래스:', baseline.names)

BASE_ALIAS = {'hardhat': 'helmet', 'helmet': 'helmet', 'nohardhat': 'head',
              'safetyvest': 'safety_vest', 'vest': 'safety_vest',
              'person': 'person', 'worker': 'person'}


def dets_baseline(res):
    out = []
    for b, c, f in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist()):
        tgt = BASE_ALIAS.get(_norm(res.names[int(c)]))
        if tgt:
            out.append((tgt, tuple(map(float, b)), float(f)))
    return out


ho_imgs = sorted((HOLDOUT / 'images' / 'val').glob('*'))[:300]
agg_new, agg_base = Counter(), Counter()
for res in m.predict(ho_imgs, imgsz=IMGSZ, conf=0.25, stream=True, verbose=False):
    agg_new.update(violations(judge_ppe(dets_from_result(res))))
for res in baseline.predict(ho_imgs, imgsz=640, conf=0.25, stream=True, verbose=False):
    agg_base.update(violations(judge_ppe(dets_baseline(res))))

print('\n같은 홀드아웃 300장')
print('  신규 yolo26 :', dict(agg_new))
print('  현행 vyra   :', dict(agg_base))
print('\n※ 위반 건수만으로는 우열을 못 가린다. 정답 라벨과 대조한 판정 정확도로 결론낼 것.')

## 10. 내보내기 → Jetson 이관

**코랩에서 TensorRT 엔진을 만들지 않는다.** 엔진은 기기·TensorRT 버전 종속이라 Jetson 에서 못 쓴다.

젯슨 실측(2026-07-27) — `yolo26s` **27.0 fps** / `yolo26m` **17.1 fps** (TensorRT FP16 @960).
둘 다 목표선(≥10 fps)을 넘겼으므로 정확도만 보고 고르면 된다.

In [ ]:
out = DRIVE / 'weights'
out.mkdir(exist_ok=True)
dst = out / f'ppe_{RUN_NAME}.pt'
shutil.copy2(best, dst)
print('저장:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')
if NC:
    print('\n⚠️ SH17(CC BY-NC-SA) · AI Hub(학습용 한정) 포함. 상용 빌드에 반입하지 말 것.')

print(f'''
다음 — Jetson Orin Nano Super 4GB (2026-07-27 정비 완료)

  # 이미 처리됨: 헤드리스 · 스왑 8GB · 디스크 49GB · ultralytics 8.4.107 · MAXN_SUPER
  # ★ 엔진 빌드 전 GUI 를 내릴 것 — 켜두면 TensorRT 가 최적 커널을 건너뛴다(실증됨)
  sudo systemctl isolate multi-user.target
  pkill -f orbbec_cctv_push.py; pkill -f gopro_ppe_push.py
  sudo jetson_clocks

  python3 -c "from ultralytics import YOLO; \\
    YOLO('ppe_{RUN_NAME}.pt').export(format='engine', half=True, imgsz={IMGSZ}, workspace=2)"

  # 카메라 파이프라인을 켠 상태에서 정상상태 재측정 — 이게 진짜 숫자다
  jtop
''')